# Instalar dependências

In [ ]:
!pip install -q unsloth transformers datasets accelerate bitsandbytes torch
!pip install gdown

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224

## Configuração

In [ ]:
base_model = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"

# Fase 1 - Treinar com dataset MedPT

**Objetivo:** Adaptação ao domínio médico

Esta etapa adapta o modelo ao vocabulário e ao contexto da área médica.

Utiliza 6400 registros do dataset AKCIT/MedPT do Hugging Face.
Sendo 400 registros para cada uma destas especialidades: ['Psicólogo', 'Ginecologista', 'Urologista', 'Dermatologista', 'Ortopedista - traumatologista', 'Oftalmologista', 'Psiquiatra', 'Dentista', 'Psicólogo, Psicanalista', 'Otorrino', 'Neurologista', 'Pediatra', 'Psicanalista, Psicólogo', 'Cirurgião do aparelho digestivo, Cirurgião geral', 'Nutricionista', 'Cardiologista'].

Este dataset possui perguntas e respostas de diversas especicialidades da medicina.

## Carregar modelo

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    max_seq_length=1024,
    load_in_4bit=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

model.config.pad_token_id = tokenizer.pad_token_id

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.8: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-14b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


## Carregar LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.8 patched 48 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


## Carregar dataset

In [ ]:
# from collections import defaultdict, Counter
# import random
# from datasets import load_dataset

# dataset_base = load_dataset("AKCIT/MedPT", split="train")

# FIELD = "medical_specialty"  # ajuste aqui se necessário
# TOP_K = 16
# SAMPLES_PER_CLASS = 400
# SEED = 42

# # 1. Contagem por especialidade
# counts = Counter(dataset_base[FIELD])

# # 2. Top 5 especialidades
# top_specialties = [s for s, _ in counts.most_common(TOP_K)]
# print("Top especialidades:", top_specialties)

# # 3. Agrupar índices por especialidade (somente top 5)
# indices_by_specialty = defaultdict(list)

# for idx, item in enumerate(dataset_base):
#     specialty = item[FIELD]
#     if specialty in top_specialties:
#         indices_by_specialty[specialty].append(idx)

# # 4. Selecionar até 400 por especialidade
# random.seed(SEED)
# selected_indices = []

# for specialty, indices in indices_by_specialty.items():
#     random.shuffle(indices)
#     selected_indices.extend(indices[:SAMPLES_PER_CLASS])

# # 5. Criar dataset final
# dataset = dataset_base.select(selected_indices)

# print(dataset[0])
# print("Quantidade de registros:", len(dataset))

# Apenas especialidade Cardiologia

In [ ]:
from collections import defaultdict, Counter
from datasets import load_dataset, concatenate_datasets
import random

dataset_base = load_dataset("AKCIT/MedPT", split="train")

FIELD = "medical_specialty"
SEED = 42
MAX_TOTAL = 6000

random.seed(SEED)

cardio_puro = dataset_base.filter(
    lambda x: x[FIELD] == "Cardiologista"
)

cardio_misto = dataset_base.filter(
    lambda x: x[FIELD] == "Cardiologista, Médico clínico geral"
)

print("Cardiologista:", len(cardio_puro))
print("Cardio + clínico:", len(cardio_misto))

selected_indices = list(range(len(cardio_puro)))

if len(selected_indices) >= MAX_TOTAL:
    selected_indices = selected_indices[:MAX_TOTAL]
    dataset = cardio_puro.select(selected_indices)

else:
    restante = MAX_TOTAL - len(selected_indices)

    indices_misto = list(range(len(cardio_misto)))
    random.shuffle(indices_misto)

    indices_misto = indices_misto[:restante]

    dataset = concatenate_datasets([
        cardio_puro.select(selected_indices),
        cardio_misto.select(indices_misto)
    ])


print(dataset[0])
print("Quantidade de registros:", len(dataset))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/95.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/384095 [00:00<?, ? examples/s]

Filter:   0%|          | 0/384095 [00:00<?, ? examples/s]

Filter:   0%|          | 0/384095 [00:00<?, ? examples/s]

Cardiologista: 5277
Cardio + clínico: 3129
{'id': 4340, 'question': 'Boa noite, estou sentindo dores no peito e palpitação, seguido de dor de cabeça.Possuo um oxímetro que sempre monitoro minha FC. Porém ele começou a mostrar batimentos muito lentos e muito rápidos entre 11 e 250 e as vezes saturação baixa.Tenho acompanhado com cardiologista. Porém ainda não se descobriu a causa.', 'answer': 'Para o diagnóstico de arritmias que não estão presentes durante todo o tempo podemos lançar mão de outros recursos como holter de 7 dias ou monitores de eventos. Outra coisa a ser levada em consideração é se não há erro de medida do oxímetro. Se possível agende consulta com cardiologista especialista em arritmias cardíacas para melhor avaliação. Espero ter ajudado!Te convidamos para uma consulta: Consulta arritmologista - R 390Você pode reservar uma consulta através do site Doctoralia, clicando no botão agendar consulta.', 'condition': 'Cefaleia (dor de cabeça)', 'medical_specialty': 'Cardiologist

## Formatar dataset

In [ ]:
def formatting_func_medpt(examples):
    texts = []

    for q, a, prof in zip(
        examples["question"],
        examples["answer"],
        examples["medical_specialty"]
    ):

        conversation = [
            { "role": "system", "content": f"Você é um médico especialista ({prof}) respondendo perguntas clínicas." },
            { "role": "user", "content": q },
            { "role": "assistant", "content": a }
        ]
        text = tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)

    return {"text": texts}

In [ ]:
dataset = dataset.map(
    formatting_func_medpt,
    batched=True
)

dataset = dataset.remove_columns(
    ["id", "question", "answer", "condition", "medical_specialty", "question_type"]
)

print(dataset[0])

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

{'text': '<|im_start|>system\nVocê é um médico especialista (Cardiologista) respondendo perguntas clínicas.<|im_end|>\n<|im_start|>user\nBoa noite, estou sentindo dores no peito e palpitação, seguido de dor de cabeça.Possuo um oxímetro que sempre monitoro minha FC. Porém ele começou a mostrar batimentos muito lentos e muito rápidos entre 11 e 250 e as vezes saturação baixa.Tenho acompanhado com cardiologista. Porém ainda não se descobriu a causa.<|im_end|>\n<|im_start|>assistant\nPara o diagnóstico de arritmias que não estão presentes durante todo o tempo podemos lançar mão de outros recursos como holter de 7 dias ou monitores de eventos. Outra coisa a ser levada em consideração é se não há erro de medida do oxímetro. Se possível agende consulta com cardiologista especialista em arritmias cardíacas para melhor avaliação. Espero ter ajudado!Te convidamos para uma consulta: Consulta arritmologista - R 390Você pode reservar uma consulta através do site Doctoralia, clicando no botão agenda

## Treinar

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    max_seq_length=1024,
    train_on_inputs = False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=1e-4,
        num_train_epochs=1,
        logging_steps=10,
        bf16 = True,
        output_dir="models/phase1",
        optim="paged_adamw_8bit"
    )
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/6000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,000 | Num Epochs = 1 | Total steps = 750
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 137,625,600 of 14,907,659,264 (0.92% trained)


Step,Training Loss
10,2.297939
20,1.654623
30,1.603510
40,1.612631
50,1.582796
60,1.573312
70,1.595947
80,1.552504
90,1.653229
100,1.568466


TrainOutput(global_step=750, training_loss=1.5059129562377929, metrics={'train_runtime': 1984.4649, 'train_samples_per_second': 3.023, 'train_steps_per_second': 0.378, 'total_flos': 1.0397140714098278e+17, 'train_loss': 1.5059129562377929, 'epoch': 1.0})

## Salvar modelo

In [ ]:
trainer.model.save_pretrained("models/phase1")
tokenizer.save_pretrained("models/phase1")

('models/phase1/tokenizer_config.json',
 'models/phase1/chat_template.jinja',
 'models/phase1/tokenizer.json')

# Fase 2 - Treinar com Casos Clínicos

**Objetivo:** Aprendizado do formato clínico

Essa etapa ensina o modelo a interpretar sintomas e gerar análises clínicas estruturadas.

9k de registros de casos clínicos sintéticos, gerados artificialmente.

Cada exemplo contém sintomas relatados pelo médico e uma resposta estruturada com resumo clínico, hipóteses diagnósticas e exames recomendados.


In [ ]:
import gc, torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

## Carregar modelo

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="models/phase1",
    max_seq_length=1024,
    load_in_4bit=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

model.config.pad_token_id = tokenizer.pad_token_id

==((====))==  Unsloth 2026.3.8: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

unsloth/qwen2.5-14b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


## Carregar dataset

In [ ]:
import gdown
from datasets import load_dataset

# FOLDER_ID = "1iaxP8SMuNVKtxBjVnems-TnMTFrBQRw6"
# FILE_NAME = "dataset_clinical_cases_generated.jsonl"

# apenas cardiologia
FILE_ID = "1jZTUuA2YnMRK9HJQqHk8vmjh9KwCHuXv"
FILE_NAME = "dataset_casos_cardiologia.jsonl"

url = f"https://drive.google.com/uc?id={FILE_ID}"
gdown.download(url, FILE_NAME, quiet=False)
dataset = load_dataset("json", data_files=FILE_NAME, split="train")

print(dataset[0])
print("Quantidade de registros:", len(dataset))

Downloading...
From: https://drive.google.com/uc?id=1jZTUuA2YnMRK9HJQqHk8vmjh9KwCHuXv
To: /content/dataset_casos_cardiologia.jsonl
100%|██████████| 8.75M/8.75M [00:00<00:00, 39.2MB/s]


Generating train split: 0 examples [00:00, ? examples/s]

{'messages': [{'role': 'system', 'content': 'Você é um assistente médico especializado em cardiologia. Se o caso não for cardiológico, informe que está fora do escopo.'}, {'role': 'user', 'content': 'Contexto do paciente:\nPaciente F, 60 anos.\nHistórico: insuficiência cardíaca\n\nSintomas (início há 1 dia):\ndispneia aos esforços, dor torácica opressiva, síncope, tosse seca\n\nSinais vitais:\nFC 86 bpm, PA 154/64 mmHg\n\nPergunta:\nQuais exames devem ser solicitados inicialmente?'}, {'role': 'assistant', 'content': 'Resumo clínico:\nPaciente com dispneia aos esforços, dor torácica opressiva, síncope, tosse seca com início há 1 dia. Fatores de risco: insuficiência cardíaca.\n\nRaciocínio clínico:\nDor torácica típica sugere síndrome coronariana. Dispneia sugere possível insuficiência cardíaca. Síncope pode estar relacionada a arritmias ou valvopatias. Fatores de risco presentes: insuficiência cardíaca.\n\nHipótese diagnóstica principal:\nestenose aórtica\n\nDiagnósticos diferenciais:\n

## Formatar dataset

In [ ]:
print(dataset.column_names)
print(dataset[0])

['messages']
{'messages': [{'role': 'system', 'content': 'Você é um assistente médico especializado em cardiologia. Se o caso não for cardiológico, informe que está fora do escopo.'}, {'role': 'user', 'content': 'Contexto do paciente:\nPaciente F, 60 anos.\nHistórico: insuficiência cardíaca\n\nSintomas (início há 1 dia):\ndispneia aos esforços, dor torácica opressiva, síncope, tosse seca\n\nSinais vitais:\nFC 86 bpm, PA 154/64 mmHg\n\nPergunta:\nQuais exames devem ser solicitados inicialmente?'}, {'role': 'assistant', 'content': 'Resumo clínico:\nPaciente com dispneia aos esforços, dor torácica opressiva, síncope, tosse seca com início há 1 dia. Fatores de risco: insuficiência cardíaca.\n\nRaciocínio clínico:\nDor torácica típica sugere síndrome coronariana. Dispneia sugere possível insuficiência cardíaca. Síncope pode estar relacionada a arritmias ou valvopatias. Fatores de risco presentes: insuficiência cardíaca.\n\nHipótese diagnóstica principal:\nestenose aórtica\n\nDiagnósticos di

In [ ]:
def formatting_func_clinical(examples):
    texts = []
    for conversation in examples["messages"]:
        text = tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

In [ ]:
dataset = dataset.map(formatting_func_clinical, batched=True)
dataset = dataset.remove_columns(["messages"])

print(dataset[0])

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

{'text': '<|im_start|>system\nVocê é um assistente médico especializado em cardiologia. Se o caso não for cardiológico, informe que está fora do escopo.<|im_end|>\n<|im_start|>user\nContexto do paciente:\nPaciente F, 60 anos.\nHistórico: insuficiência cardíaca\n\nSintomas (início há 1 dia):\ndispneia aos esforços, dor torácica opressiva, síncope, tosse seca\n\nSinais vitais:\nFC 86 bpm, PA 154/64 mmHg\n\nPergunta:\nQuais exames devem ser solicitados inicialmente?<|im_end|>\n<|im_start|>assistant\nResumo clínico:\nPaciente com dispneia aos esforços, dor torácica opressiva, síncope, tosse seca com início há 1 dia. Fatores de risco: insuficiência cardíaca.\n\nRaciocínio clínico:\nDor torácica típica sugere síndrome coronariana. Dispneia sugere possível insuficiência cardíaca. Síncope pode estar relacionada a arritmias ou valvopatias. Fatores de risco presentes: insuficiência cardíaca.\n\nHipótese diagnóstica principal:\nestenose aórtica\n\nDiagnósticos diferenciais:\n- fibrilação atrial\n

## Treinar

In [ ]:
import os, gc, torch
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print(f"VRAM livre: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

VRAM livre: 20.3 GB


In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    max_seq_length=512,          # reduzido de 1024 → 512
    train_on_inputs=False,
    packing=True,                 # Acelerar treino
    args=TrainingArguments(
        per_device_train_batch_size=2,   # reduzido de 4 → 2
        gradient_accumulation_steps=4,   # aumentado de 2 → 4 (mantém batch efetivo = 8)
        learning_rate=5e-5,
        num_train_epochs=2,
        logging_steps=10,
        bf16=True,
        output_dir="models/final",
        optim="paged_adamw_8bit"
    )
)
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/9000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,000 | Num Epochs = 2 | Total steps = 2,250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 137,625,600 of 14,907,659,264 (0.92% trained)


Step,Training Loss
10,0.841346
20,0.264930
30,0.181028
40,0.161235
50,0.157137
60,0.146562
70,0.147148
80,0.142562
90,0.141687
100,0.141611


TrainOutput(global_step=2250, training_loss=0.13045603550804985, metrics={'train_runtime': 5934.9113, 'train_samples_per_second': 3.033, 'train_steps_per_second': 0.379, 'total_flos': 4.5553468536695194e+17, 'train_loss': 0.13045603550804985, 'epoch': 2.0})

## Salvar modelo

In [ ]:
# Merge LoRA + base e salva como modelo normal
# model = model.merge_and_unload()
# model.save_pretrained("models/final_merged")
# tokenizer.save_pretrained("models/final_merged")

# # Zip e copia pro Drive
# !cd models/final_merged && zip -rq ../medical_assistant_llm_merged.zip .
# !cp models/medical_assistant_llm_merged.zip "/content/drive/MyDrive/Colab Notebooks/FIAP - TC - Fase 3/"

In [ ]:
model.save_pretrained("models/final_saved")
tokenizer.save_pretrained("models/final_saved")

!ls "models/final_saved"

# compactar modelo
!cd models/final_saved && zip -rq ../medical_cardiology_assistant_llm_14b.zip .

!ls

adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json
dataset_casos_cardiologia.jsonl  models
drive				 sample_data
huggingface_tokenizers_cache	 unsloth_compiled_cache


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# salvar no drive
!cp "models/medical_cardiology_assistant_llm_14b.zip" "/content/drive/MyDrive/Colab Notebooks/FIAP - TC - Fase 3/"

Mounted at /content/drive
cp: cannot stat 'models/medical_cardiology_assistant_llm_14b.zip': No such file or directory
